# Silver Layer — Metadata Transformation

**Catalog:** metadata_governance

**Schema:** silver

**Table:** silver_metadata

**Layer:** Silver (cleaned, standardized, and validated data with quality rules applied)

**Source:** Bronze Layer Table — metadata_governance.bronze.bronze_metadata

**Purpose:** The Silver layer processes raw Bronze data by applying data cleaning, standardization, and validation rules to produce a trusted dataset ready for analytics and reporting.

## Step 1 — Read Metadata from Bronze Layer

Read the metadata dataset from the Bronze layer and verify that all records are successfully loaded before transformation.

In [0]:
bronze_df = spark.table("metadata_governance.bronze.bronze_metadata")

bronze_count = bronze_df.count()
print(f"Bronze rows loaded: {bronze_count}")

## Step 2 — Clean and Standardize Metadata

Standardize text fields by removing leading and trailing spaces and converting values to lowercase to ensure consistent formatting across the dataset.

In [0]:
from pyspark.sql.functions import col, trim, lower

text_cols = [
    "column_name","column_desc", "term_name", "term_description",
    "security_classification",
    "term_subdomain", "data_steward","table_name", "table_desc",
    "table_owner_in_source", "schema_name", "database_name",
    "system_name", "location", "tag_name", "tag_value", "certification_level"
]

df_cleaned = bronze_df

for c in text_cols:
    if c in df_cleaned.columns:
        df_cleaned = df_cleaned.withColumn(
            c,
            lower(trim(col(c)))
        )

print(f"Cleaned rows: {df_cleaned.count()}")

## Step 3 — Apply Data Quality Validation Rules

Validate required metadata fields and separate invalid records into a quarantine dataset while retaining valid records for further processing.

In [0]:
from pyspark.sql.functions import col, isnull, when, concat_ws, lit

# INVALID ROWS
df_invalid = df_cleaned.filter(
    isnull(col("table_name")) | (col("table_name") == "") |
    isnull(col("schema_name")) | (col("schema_name") == "") |
    isnull(col("database_name")) | (col("database_name") == "")
).withColumn(
    "failure_reason",
    concat_ws(", ",
        when(isnull(col("table_name")) | (col("table_name") == ""),lit("table_name is null")),
        when(isnull(col("schema_name")) | (col("schema_name") == ""),lit("schema_name is null")),
        when(isnull(col("database_name")) | (col("database_name") == ""), lit("database_name is null"))
    )
)

# VALID ROWS
df_valid = df_cleaned.filter(
    col("table_name").isNotNull() & (col("table_name") != "") &
    col("schema_name").isNotNull() & (col("schema_name") != "") &
    col("database_name").isNotNull() & (col("database_name") != "")
)

# COUNTS
valid_count = df_valid.count()
invalid_count = df_invalid.count()

print(f"Valid rows: {valid_count}")
print(f"Invalid rows: {invalid_count}")
print(f"Total check: {valid_count + invalid_count}")
assert valid_count + invalid_count == bronze_count, "Row count mismatch"

In [0]:
display(df_invalid.groupBy("failure_reason").count())

## Step 4 — Save Validated Metadata to Silver Layer

Store the validated metadata dataset in the Silver layer as a trusted source for analytics, reporting, and governance activities.



In [0]:
df_valid.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("metadata_governance.silver.silver_metadata")

print(f"Silver table written: {df_valid.count()} rows")

## Step 5 — Save Quarantine Records and Generate Compliance Alert

Store invalid records in a quarantine table, generate a failure summary, and create a compliance alert when validation rules are violated.

In [0]:

if invalid_count > 0:
    
    df_invalid.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("metadata_governance.silver.bronze_metadata_quarantine")

    print(f"Alert: {invalid_count} rows moved to quarantine!")

    print("Failure breakdown:")
    display(
        df_invalid.groupBy("failure_reason")
        .count()
        .orderBy("count", ascending=False)
    )

    alert_msg = f"[COMPLIANCE ALERT] {invalid_count} rows failed validation. Table: silver_metadata_quarantine. Date: {spark.sql('SELECT current_date()').collect()[0][0]}"

    print(alert_msg)

else:
    print("All rows passed validation")

print("Notebook execution completed successfully.")

## Step 6 — Validate Silver Output

> Verify that the Silver table was successfully created by checking the row count, schema, and sample records.


In [0]:
silver_df = spark.table("metadata_governance.silver.silver_metadata")
print(f"Silver final row count: {silver_df.count()}")
silver_df.printSchema()
display(silver_df.limit(5))

# Step 7 - Create Metadata Entities

In [0]:
silver_df = spark.table("metadata_governance.silver.silver_metadata")
print(silver_df.count())  

In [0]:
# ─────────────────────────────────────────────
# TASK 6 — Create Metadata Entities
# Input  : silver.silver_metadata 
# Output : 9 entity tables in silver schema
# ─────────────────────────────────────────────

from pyspark.sql.functions import col, when, initcap, expr, lit
from pyspark.sql import functions as F

silver_df = spark.table("metadata_governance.silver.silver_metadata")
input_count = silver_df.count()

print(f"Silver input rows: {input_count}")
print(f"Quarantined rows: 10000 - {input_count} = {10000 - input_count}")

## 7.1 Create System Entity Table
Created a table for the system entity by selecting system_id, system_name, and location from the Silver dataset. Rows are deduplicated on system_id to ensure each system appears only once. The result is written to metadata_governance.silver.dim_system as a Delta table, and the final row count is printed to confirm successful population.

In [0]:
dim_system = silver_df.select(
    "system_id",
    "system_name",
    "location"
).dropDuplicates(["system_id"])

dim_system.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.silver.dim_system")

cnt = spark.table("metadata_governance.silver.dim_system").count()
print(f"dim_system rows written: {cnt}")

##7.2 — Create Database Entity Table


Create a table for the database entity by selecting database_id, database_name, and system_id from the Silver dataset.  The result is written to metadata_governance.silver.dim_database as a Delta table, and the final row count is printed to confirm successful population.

In [0]:
dim_database = silver_df.select(
    "database_id",
    "database_name",
    "system_id"      
).dropDuplicates(["database_id"])

dim_database.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.silver.dim_database")

cnt = spark.table("metadata_governance.silver.dim_database").count()
print(f"dim_database rows written: {cnt}")

##7.3 — Create Schema Entity Table

Create a table for the schema entity by selecting schema_id, schema_name, and database_id from the Silver dataset.The result is written to metadata_governance.silver.dim_schema as a Delta table, and the final row count is printed to confirm successful population.

In [0]:
dim_schema = silver_df.select(
    "schema_id",
    "schema_name",
    "database_id"    
).dropDuplicates(["schema_id"])

dim_schema.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.silver.dim_schema")

cnt = spark.table("metadata_governance.silver.dim_schema").count()
print(f"dim_schema rows written: {cnt}")

##7.4 — Create Table Entity Table

Create a table for the table entity by selecting table_id, table_name, table_desc, table_owner_in_source, certification_level, total_record_count, invalid_record_count, and schema_id from the Silver dataset.The result is written to metadata_governance.silver.dim_table as a Delta table, and the final row count is printed to confirm successful population.

In [0]:
dim_table = silver_df.select(
    "table_id",
    "table_name",
    "table_desc",
    "table_owner_in_source",
    "certification_level",     
    "total_record_count",
    "invalid_record_count",
    "schema_id"                
).dropDuplicates(["table_id"])

dim_table.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.silver.dim_table")

cnt = spark.table("metadata_governance.silver.dim_table").count()
print(f"dim_table rows written: {cnt}")

##7.5 Create Column Entity Table
Create a table for the column entity by selecting column_id, column_name, column_desc, data_steward, security_classification, pii_flag, critical_data_element_flag, and table_id from the Silver dataset. The result is written to metadata_governance.silver.dim_column as a Delta table, and the final row count is printed to confirm successful population.

In [0]:
dim_column = silver_df.select(
    "column_id",
    "column_name",
    "column_desc",
    "data_steward",
    "security_classification",   
    "pii_flag",
    "critical_data_element_flag",
    "table_id"                  
).dropDuplicates(["column_id"])

dim_column.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.silver.dim_column")

cnt = spark.table("metadata_governance.silver.dim_column").count()
print(f"dim_column rows written: {cnt}")

##7.6 — Create Term Entity Table

Create a table for the business glossary term entity by selecting term_name, term_description, and term_subdomain from the Silver dataset.surrogate term_id is generated using UUID to uniquely identify each term. The result is written to metadata_governance.silver.dim_term as a Delta table, and the final row count is printed to confirm successful population.

In [0]:
dim_term = silver_df.select(
    "term_name",
    "term_description",
    "term_subdomain"
).dropDuplicates(["term_name"]) \
 .withColumn("term_id", expr("uuid()"))

dim_term.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.silver.dim_term")

cnt = spark.table("metadata_governance.silver.dim_term").count()
print(f"dim_term rows written: {cnt}")

## 7.8 — Create Column-Term Bridge Table
Build the column → term bridge table by joining dim_column and dim_term

In [0]:
dim_term_ref = spark.table("metadata_governance.silver.dim_term") \
    .select("term_id", "term_name")

bridge_col_term = silver_df.select(
    "column_id",
    "term_name"
).dropna(subset=["term_name"]) \
 .join(dim_term_ref, on="term_name", how="inner") \
 .select("column_id", "term_id") \
 .dropDuplicates()

bridge_col_term.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.silver.bridge_col_term")

cnt = spark.table("metadata_governance.silver.bridge_col_term").count()
print(f"bridge_col_term rows written: {cnt}")

In [0]:
# Step 3: Verify — 0 orphans expected
orphan_term = spark.table("metadata_governance.silver.bridge_col_term") \
    .join(spark.table("metadata_governance.silver.dim_term"),
          on="term_id", how="left_anti").count()

print(f"bridge_col_term orphan term_ids: {orphan_term}  {'✅ OK' if orphan_term == 0 else '❌ MISMATCH — do not proceed'}")


##7.7 — Create Tag Entity Table


Create a table for the domain tag entity by selecting tag_name and tag_value from the Silver dataset. A surrogate tag_id is generated using UUID to uniquely identify each tag. The result is written to metadata_governance.silver.dim_tag as a Delta table, and the final row count is printed to confirm successful population.

In [0]:
dim_tag = silver_df.select(
    "tag_name",
    "tag_value"
).filter(
    col("tag_value").isNotNull() & (col("tag_value") != "") 
).dropDuplicates(["tag_name", "tag_value"]) \
 .withColumn("tag_id", expr("uuid()"))

dim_tag.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.silver.dim_tag")

cnt = spark.table("metadata_governance.silver.dim_tag").count()
print(f"dim_tag rows written: {cnt}")

##7.9 — Create Column-Tag Bridge Table

Build the column → tag bridge table by joining dim_column and dim_tag 

In [0]:
dim_tag_ref = spark.table("metadata_governance.silver.dim_tag") \
    .select("tag_id", "tag_name", "tag_value")

bridge_col_tag = silver_df.select(
    "column_id",
    "tag_name",
    "tag_value"
).dropna(subset=["tag_value"]) \
 .join(dim_tag_ref, on=["tag_name", "tag_value"], how="inner") \
 .select("column_id", "tag_id") \
 .dropDuplicates()

bridge_col_tag.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.silver.bridge_col_tag")

cnt = spark.table("metadata_governance.silver.bridge_col_tag").count()
print(f"bridge_col_tag rows written: {cnt}")


In [0]:
orphan_tag = spark.table("metadata_governance.silver.bridge_col_tag") \
    .join(spark.table("metadata_governance.silver.dim_tag"),
          on="tag_id", how="left_anti").count()

print(f"bridge_col_tag orphan tag_ids: {orphan_tag}  {'✅ OK' if orphan_tag == 0 else '❌ MISMATCH — do not proceed'}")


##7.10 — Entity Row Count Summary

Print a row count summary for all 9 entity tables alongside the Silver input count and quarantine totals, confirming each table has been successfully populated from the Silver dataset.

In [0]:
tables = [
    ("dim_system",      "system / catalog entity"),
    ("dim_database",    "database"),
    ("dim_schema",      "schema"),
    ("dim_table",       "table"),
    ("dim_column",      "column — governance entity"),
    ("dim_term",        "business glossary terms"),
    ("dim_tag",         "domain tags"),
    ("bridge_col_term", "column → term"),
    ("bridge_col_tag",  "column → tag"),
]

print("=" * 55)
print("  TASK 6 — Entity Row Count Summary")
print("=" * 55)
for tbl, desc in tables:
    cnt = spark.table(f"metadata_governance.silver.{tbl}").count()
    print(f"  {tbl:<22} {cnt:>6} rows  ({desc})")
print("=" * 55)
print(f"  Input Silver rows:          9543")
print(f"  Quarantined (not in scope):  457")
print(f"  Original Bronze total:     10000")
print("=" * 55)

## 7.11 — Verify Silver Entity Tables Exist

Verify that all core entity tables (dim_column, dim_table, dim_schema, dim_database, dim_system) exist and contain data before proceeding to the next step.

In [0]:
# Verify all Silver inputs exist
tables = ["dim_column", "dim_table", "dim_schema", "dim_database", "dim_system"]
for t in tables:
    cnt = spark.table(f"metadata_governance.silver.{t}").count()
    print(f"{t}: {cnt} rows")

##Step 8 - Validate Entity Relationships

##8.1-Load Entity Tables for Relationship Validation

Load all five core entity tables from Unity Catalog and print their row counts to confirm they are ready for FK join validation.

In [0]:
# ─────────────────────────────────────────────────────────
# Define Entity Relationships
# Catalog : metadata_governance
# Schema  : silver
# Purpose : Confirm all FK joins work and no rows are
#           left unmatched between entity tables
# ─────────────────────────────────────────────────────────

dim_system   = spark.table("metadata_governance.silver.dim_system")
dim_database = spark.table("metadata_governance.silver.dim_database")
dim_schema   = spark.table("metadata_governance.silver.dim_schema")
dim_table    = spark.table("metadata_governance.silver.dim_table")
dim_column   = spark.table("metadata_governance.silver.dim_column")

print("All entity tables loaded successfully")
print(f"  dim_system   : {dim_system.count()} rows")
print(f"  dim_database : {dim_database.count()} rows")
print(f"  dim_schema   : {dim_schema.count()} rows")
print(f"  dim_table    : {dim_table.count()} rows")
print(f"  dim_column   : {dim_column.count()} rows")

## 8.2 — Join 1: Schema → Database → System

Validate the relationship between dim_schema, dim_database, and dim_system using left joins on database_id and system_id. Unmatched rows are counted and printed to confirm no rows are left without a matching database or system record.

In [0]:
# ── JOIN 1: dim_schema → dim_database → dim_system 

join1 = dim_schema \
    .join(dim_database, on="database_id", how="left") \
    .join(dim_system,   on="system_id",   how="left") \
    .select(
        dim_schema["schema_id"],
        dim_schema["schema_name"],
        dim_database["database_name"],
        dim_system["system_name"],
        dim_system["location"]
    )

total         = join1.count()
unmatched_db  = join1.filter(dim_database["database_name"].isNull()).count()
unmatched_sys = join1.filter(dim_system["system_name"].isNull()).count()

print("=" * 55)
print("  JOIN 1: dim_schema → dim_database → dim_system")
print("=" * 55)
print(f"  Total rows returned     : {total}")
print(f"  Unmatched (no database) : {unmatched_db}")
print(f"  Unmatched (no system)   : {unmatched_sys}")
print("=" * 55)
print("\nSample results:")
display(join1.limit(10))

## 8.3 — Join 2: Table → Schema

Validate the relationship between dim_table and dim_schema using a left join on schema_id. Unmatched rows are counted and printed to confirm every table has a matching schema record.

In [0]:
# ── JOIN 2: dim_table → dim_schema 

join2 = dim_table \
    .join(dim_schema, on="schema_id", how="left") \
    .select(
        dim_table["table_id"],
        dim_table["table_name"],
        dim_table["certification_level"],
        dim_schema["schema_name"],
        dim_schema["database_id"]
    )

total     = join2.count()
unmatched = join2.filter(dim_schema["schema_name"].isNull()).count()

print("=" * 55)
print("  JOIN 2: dim_table → dim_schema")
print("=" * 55)
print(f"  Total rows returned : {total}")
print(f"  Unmatched rows      : {unmatched}")
print("=" * 55)
print("\nSample results:")
display(join2.limit(10))

##JOIN 3: bridge_col_term → dim_column + dim_term
Validates the M:M column-to-term relationship through the bridge.
 

In [0]:
 
join3 = bridge_col_term \
    .join(dim_column, on="column_id", how="left") \
    .join(dim_term,   on="term_id",   how="left") \
    .select(
        dim_column["column_id"],
        dim_column["column_name"],
        dim_term["term_name"],
        dim_term["term_subdomain"]
    )
 
total     = join3.count()
unmatched = join3.filter(dim_term["term_name"].isNull()).count()
 
print("=" * 55)
print("  JOIN 3: bridge_col_term → dim_column + dim_term")
print("=" * 55)
print(f"  Total rows returned : {total}")
print(f"  Unmatched rows      : {unmatched}")
print("=" * 55)
print("\nSample results:")
display(join3.limit(10))

##JOIN 4: bridge_col_tag → dim_column + dim_tag
Validates the M:M column-to-tag relationship through the bridge.


In [0]:

join4 = bridge_col_tag \
    .join(dim_column, on="column_id", how="left") \
    .join(dim_tag,    on="tag_id",    how="left") \
    .select(
        "column_id",
        "column_name",
        "tag_name",
        "tag_value"
    )

total     = join4.count()
unmatched = join4.filter(col("tag_name").isNull()).count()

print("=" * 55)
print("  JOIN 4: bridge_col_tag → dim_column + dim_tag")
print("=" * 55)
print(f"  Total rows returned : {total}")
print(f"  Unmatched rows      : {unmatched}")
print("=" * 55)
print("\nSample results:")
display(join4.limit(10))

## Step 8.4 — Validate All FK Relationships

Run anti-join checks across all four FK relationships (database → system, schema → database, table → schema, column → table) to detect any orphan rows. Each check prints PASS if no unmatched rows are found, or FAIL with the orphan count. A final summary confirms whether all FK relationships are valid.

In [0]:
# ── ORPHAN CHECK — left_anti on every FK relationship ────
# Returns rows that have NO match in the parent table.
# Expected: 0 orphans on every check.
 
checks = [
    # Core hierarchy FK checks
    ("dim_database has valid system_id",
     dim_database.join(dim_system,   on="system_id",   how="left_anti")),
 
    ("dim_schema has valid database_id",
     dim_schema.join(dim_database,   on="database_id", how="left_anti")),
 
    ("dim_table has valid schema_id",
     dim_table.join(dim_schema,      on="schema_id",   how="left_anti")),
 
    ("dim_column has valid table_id",
     dim_column.join(dim_table,      on="table_id",    how="left_anti")),
 
    
    ("bridge_col_term has valid column_id",
     bridge_col_term.join(dim_column, on="column_id",  how="left_anti")),
 
    ("bridge_col_term has valid term_id",
     bridge_col_term.join(dim_term,   on="term_id",    how="left_anti")),
 
    ("bridge_col_tag has valid column_id",
     bridge_col_tag.join(dim_column,  on="column_id",  how="left_anti")),
 
    ("bridge_col_tag has valid tag_id",
     bridge_col_tag.join(dim_tag,     on="tag_id",     how="left_anti")),
]
 
print("=" * 55)
print("  ORPHAN CHECK — All FK Relationships")
print("=" * 55)
all_passed = True
for label, anti_df in checks:
    orphan_count = anti_df.count()
    status = "PASS" if orphan_count == 0 else "FAIL"
    if orphan_count > 0:        
        all_passed = False
    print(f"  {status}  {label}: {orphan_count} orphans")  
 
print("=" * 55)
if all_passed:
    print("  All FK relationships valid — no orphaned rows.")
else:
    print("  WARNING: Some rows have no matching parent record.")
print("=" * 55)